# Edge SQ-GAN fine-tune v1 (teammate hand-off)

Step `edge-sqgan-finetune-v1`. Thin wrapper; logic lives in `Codec.edge.sqgan_finetune`.


In [ ]:
import os, subprocess
REPO, URL, BRANCH = "/kaggle/working/repo", "https://github.com/saltypal/SemanticSchedulerEnd2End.git", "main"
if os.path.isdir(REPO + "/.git"):
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/" + BRANCH], check=True)
else:
    subprocess.run(f"git clone --depth 1 --branch {BRANCH} {URL} {REPO}", shell=True, check=True)
GIT_SHA = subprocess.check_output(f"git -C {REPO} rev-parse HEAD", shell=True, text=True).strip()
print("commit", GIT_SHA)

In [ ]:
%cd /kaggle/working/repo
!pip install -q -e . pytorch-lightning==1.9.5 torchmetrics==0.11.4 transformers==4.46.3 scipy omegaconf pytorch-msssim==1.0.0 lpips==0.1.4 PyYAML einops

## Trainer source

Prints whether Semantic_Lossless came from the pinned GitHub commit or the Kaggle fallback.


In [ ]:
!python -u -m Codec.edge.sqgan_finetune --config config/edge/sqgan_finetune.yaml --stage source

In [ ]:
import time; T0 = time.time()

In [ ]:
!python -u -m Codec.edge.sqgan_finetune --config config/edge/sqgan_finetune.yaml

In [ ]:
import datetime, json, os, platform, subprocess, time
STEP, ARGS = "edge-sqgan-finetune-v1", ['--config', 'config/edge/sqgan_finetune.yaml']
def _gpu():
    try:
        return subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
    except Exception:
        return None
try:
    import torch; torch_version = torch.__version__
except Exception:
    torch_version = None
out = "/kaggle/working/runs/" + STEP
os.makedirs(out, exist_ok=True)
manifest = {"step": STEP, "entrypoint": "Codec.edge.sqgan_finetune", "args": ARGS, "git_sha": GIT_SHA,
            "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "python": platform.python_version(),
            "torch": torch_version, "gpu": _gpu(), "wall_time_s": time.time() - T0}
json.dump(manifest, open(out + "/run_manifest.json", "w"), indent=2)
print(manifest)